# Creating Loan Offers with Feature Engineering
## Solution

**Short name (GitHub):** `LoanOrig`

Hold-out 184 of 920, `random_state=42`, leakage-safe encode, default 100-tree forest.

**MAE ≈ $5,153**, **RMSE ≈ $9,754**, **R² ≈ 0.982**. Mean-offer baseline MAE ≈ **$54,658**. LinearRegression MAE ≈ **$8,250**, R² ≈ **0.972**. Train R² ≈ 0.997.

`requested_usd` corr with offer is 0.98 — the teaching policy is roughly min(ask, capacity). That is a feature, not a bug. Practice cell 2 drops the ask on purpose.


## Inline cheat-sheet

| After clean | Role |
|-------------|------|
| `requested_usd` | applicant ask; impurity ~0.96 |
| `note_rate` | next impurity block (price vs size trade) |
| `annual_income_usd` / `net_capacity` | capacity side; raw corr ~0.37 |
| `term_months` / `down_payment_usd` | product geometry (mortgage tail) |
| high-card | `branch` (8), `product_class` (8), `employment_status` (5) |
| one-hot | channel, purpose, complexity, timing, digital, existing, resident, collateral |


## Flowchart

![flow](loanorig_flowchart.png)


## 0. Packages


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
sns.set_theme(style="whitegrid")


## 1. Load


In [ ]:
df = pd.read_csv("data/loan_applications.csv")
print(df.head()); print(df.shape); print(df.info()); print(df.nunique())
print(df["offer_amount"].describe())


## 2. Money + net capacity


In [ ]:
money_cols = ["annual_income_usd","other_debt_usd","requested_usd","down_payment_usd","reserves_usd"]
for col in money_cols:
    df[col] = (df[col].astype(str).str.replace("$","",regex=False)
               .str.replace(",","",regex=False).astype(float))
df["net_capacity"] = df["annual_income_usd"] - df["other_debt_usd"]
print(df[money_cols + ["net_capacity"]].head())


## 3. Artifacts


In [ ]:
df["note_rate"] = df["note_rate"].astype(str).str.replace("%","",regex=False).astype(float) / 100
df["bureau_stars"] = (df["bureau_stars"].astype(str)
    .str.replace(" stars","",regex=False).str.replace(" star","",regex=False).astype(int))
df["orig_year_n"] = (df["orig_vintage"].astype(str)
    .str.replace("Not Available","0",regex=False).str.replace(" VY","",regex=False).astype(int))
df = df.drop(columns=["orig_vintage"])
df["borrowers"] = df["borrowers"].astype(str).str.extract(r"(\d+)", expand=False).astype(int)
df["term_months"] = df["term_months"].astype(str).str.replace("-month","",regex=False).astype(int)
print(df[["note_rate","bureau_stars","orig_year_n","borrowers","term_months"]].head())


## 4. EDA


In [ ]:
numeric_df = df.select_dtypes(include="number")
print(numeric_df.corr()["offer_amount"].sort_values(ascending=False).round(3))
plt.figure(figsize=(11,8))
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation heatmap"); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,5)); sns.boxplot(x="borrowers", y="offer_amount", data=df); plt.show()
plt.figure(figsize=(8,5)); sns.boxplot(x="bureau_stars", y="offer_amount", data=df); plt.show()
plt.figure(figsize=(8,5)); sns.histplot(df["offer_amount"], bins=40, kde=True)
plt.axvline(df["offer_amount"].median(), color="crimson", ls="--")
plt.axvline(df["offer_amount"].mean(), color="navy", ls=":"); plt.show()


Expected: `requested_usd` 0.98, `down_payment` / `term` ride the mortgage tail, `note_rate` negative (−0.32), income/capacity ~0.37.


## 5–7. Encode, fit, evaluate (lesson + safe)


In [ ]:
work = df.copy()
cat_cols = work.select_dtypes(include=["object"]).columns.tolist()
print({c: work[c].nunique() for c in cat_cols})
for col in cat_cols:
    if work[col].nunique() < 5:
        dummies = pd.get_dummies(work[col], prefix=col, drop_first=True)
        work = pd.concat([work, dummies], axis=1); work.drop(columns=[col], inplace=True)
    else:
        work[col] = work[col].map(work.groupby(col)["offer_amount"].mean())
X, y = work.drop(columns=["offer_amount"]), work["offer_amount"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
base = mean_absolute_error(y_test, np.full_like(y_test, y_train.mean(), dtype=float))
print(f"MAE {mae:,.2f}  RMSE {rmse:,.2f}  R² {r2:.4f}  baseline {base:,.2f}")
print("train R²", round(r2_score(y_train, rf_model.predict(X_train)), 4))
imp = rf_model.feature_importances_; names = np.array(X_train.columns)
top = np.argsort(imp)[-5:][::-1]
print(pd.Series(imp[top], index=names[top]))
plt.figure(figsize=(8,5)); sns.barplot(x=imp[top], y=names[top], color="#1F4E79"); plt.show()
plt.figure(figsize=(6,6)); plt.scatter(y_test, y_pred, s=28, alpha=0.55)
lo,hi=min(y_test.min(),y_pred.min()),max(y_test.max(),y_pred.max())
plt.plot([lo,hi],[lo,hi],"r--"); plt.title(f"MAE={mae:,.0f}  R²={r2:.3f}"); plt.show()


### Reference numbers (safe encode, rs=42)

| Metric | Value |
|--------|-------|
| Test MAE | **$5,153** |
| Test RMSE | **$9,754** |
| Test R² | **0.982** |
| Train R² | 0.997 |
| Mean baseline MAE | **$54,658** |
| LinearRegression | MAE $8,250 / R² 0.972 |

Forest is already flat by ~25–50 trees. Permutation still puts `requested_usd` first; `note_rate` is the only other material column.


## 8–9. Alternates and practice


In [ ]:
y_safe = df["offer_amount"]; X_raw = df.drop(columns=["offer_amount"])
Xtr, Xte, ytr, yte = train_test_split(X_raw, y_safe, test_size=0.2, random_state=42)
cat = Xtr.select_dtypes(include=["object"]).columns.tolist()
low = [c for c in cat if df[c].nunique() < 5]
high = [c for c in cat if df[c].nunique() >= 5]
Xtr_enc, Xte_enc = Xtr.copy(), Xte.copy()
for col in high:
    means = pd.concat([Xtr_enc[col], ytr], axis=1).groupby(col)[ytr.name].mean()
    Xtr_enc[col] = Xtr_enc[col].map(means)
    Xte_enc[col] = Xte_enc[col].map(means).fillna(ytr.mean())
for col in low:
    dtr = pd.get_dummies(Xtr_enc[col], prefix=col, drop_first=True)
    dte = pd.get_dummies(Xte_enc[col], prefix=col, drop_first=True)
    dte = dte.reindex(columns=dtr.columns, fill_value=0)
    Xtr_enc = pd.concat([Xtr_enc.drop(columns=[col]), dtr], axis=1)
    Xte_enc = pd.concat([Xte_enc.drop(columns=[col]), dte], axis=1)
rf_safe = RandomForestRegressor(random_state=42).fit(Xtr_enc, ytr)
yp = rf_safe.predict(Xte_enc)
print("SAFE", round(mean_absolute_error(yte, yp),2), round(r2_score(yte, yp),4))
print("LR", round(mean_absolute_error(yte, LinearRegression().fit(Xtr_enc,ytr).predict(Xte_enc)),2))

# drop the ask
drop_ask = [c for c in Xtr_enc.columns if c == "requested_usd"]
rf2 = RandomForestRegressor(random_state=42).fit(Xtr_enc.drop(columns=drop_ask), ytr)
print("no-ask MAE", round(mean_absolute_error(yte, rf2.predict(Xte_enc.drop(columns=drop_ask))),2),
      "R²", round(r2_score(yte, rf2.predict(Xte_enc.drop(columns=drop_ask))),4))

# log target
rf_log = RandomForestRegressor(random_state=42)
rf_log.fit(Xtr_enc, np.log1p(ytr))
print("log MAE", round(mean_absolute_error(yte, np.expm1(rf_log.predict(Xte_enc))),2))

tail = yte >= 80000
print("tail share", float(tail.mean()), "tail MAE",
      round(mean_absolute_error(yte[tail], yp[tail]),2) if tail.any() else None)

stat = np.minimum(Xte_enc["requested_usd"], 0.35 * Xte_enc["annual_income_usd"] * Xte_enc["term_months"] / 12)
print("policy-rebuild MAE", round(mean_absolute_error(yte, stat),2))


## 10. Simulation


In [ ]:
N_EST, MAX_DEPTH, NOISE_SD, SUBSAMPLE, RANDOM_STATE = 100, None, 0, 1.0, 42
rng = np.random.default_rng(RANDOM_STATE)
n = max(int(len(Xtr_enc)*SUBSAMPLE), 20)
idx = rng.choice(len(Xtr_enc), size=n, replace=False)
y_s = ytr.iloc[idx].astype(float) + rng.normal(0, NOISE_SD, size=n)
sim = RandomForestRegressor(n_estimators=N_EST, max_depth=MAX_DEPTH, random_state=RANDOM_STATE)
sim.fit(Xtr_enc.iloc[idx], y_s)
print("sim MAE", round(mean_absolute_error(yte, sim.predict(Xte_enc)),2),
      "R²", round(r2_score(yte, sim.predict(Xte_enc)),4))


Dropping `requested_usd` is the useful stress test: R² falls from the high 0.98s toward the capacity block (income, term, product, rate). That is the model an underwriter would call “what should we offer if the ask is missing.” Inquiry and trade counts barely move permutation importance — they are file texture, not an origination lever.
